# DuckDB on S3

Write the Iris dataset to a Parquet file in S3 and query it back with SQL.

DuckDB does not need a database server. `duckdb.connect()` opens a temporary in-memory database inside this notebook process, and it is gone when the kernel stops. The data that persists is the Parquet file in your S3 bucket. DuckDB reads and writes it through its `httpfs` extension.

The S3 endpoint and credentials come from the `S3_ENDPOINT`, `S3_USE_HTTPS`, `AWS_ACCESS_KEY_ID`, and `AWS_SECRET_ACCESS_KEY` environment variables. prokube adds them to labs through a workspace configuration (a Kubeflow PodDefault). If they are missing, ask your administrator. See [File Storage](https://prokube.ai/docs/1.9/platform/file_storage.html) in the platform documentation.

In [ ]:
# DuckDB is not included in the pk notebook image.
!pip install duckdb

In [ ]:
import duckdb
import os 

conn = duckdb.connect()
conn.execute("INSTALL httpfs")
conn.execute("LOAD httpfs")
conn.execute("SET s3_url_style='path'")
conn.execute(f"SET s3_endpoint='{os.environ['S3_ENDPOINT']}'")
conn.execute(f"SET s3_use_ssl = {'false' if os.environ['S3_USE_HTTPS'] == '0' else 'true'}")

DuckDB reads the credentials from the `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` environment variables automatically, so they do not need to be set here.

## Iris dataset in S3 Parquet

Write the Iris dataset to an S3-backed Parquet file and read it back with DuckDB.

In [ ]:
from sklearn.datasets import load_iris

bucket = "<bucketname>"
key = "storage-examples/iris.parquet"

if not bucket or bucket == "<bucketname>":
    raise ValueError("Set bucket to your S3 bucket name.")

In [ ]:
iris = load_iris(as_frame=True).frame
s3_path = f"s3://{bucket}/{key}"

conn.register("iris", iris)
conn.execute(
    f"COPY iris TO '{s3_path}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)"
)

read_back = conn.sql(f"SELECT * FROM read_parquet('{s3_path}')").df()
read_back.head()